In [1]:
#undef __noinline__

# Thread Cooperation - The Grid-Stride Loop

In the previous notebooks each GPU thread handled exactly one array element.
That only works when you launch exactly as many threads as you have data points.

A more flexible pattern is the **grid-stride loop**: launch a fixed number of threads
(typically chosen to saturate the GPU) and have each thread walk through the array
in steps equal to the total number of threads in the grid.

If the grid has `T` threads total and the array has `N` elements, thread `i` handles
indices `i`, `i + T`, `i + 2T`, and so on until it runs off the end.

This means:
- You never need to know N at launch time
- You can reuse the same launch config for any array size
- Each thread does a balanced share of the work

In [2]:
#include <iostream>
#define N (64 * 1024)

__global__ void strided_add(int *a, int *b, int *c) {
    // Global thread ID across the entire grid
    int tid = threadIdx.x + blockIdx.x * blockDim.x;
    // Total number of threads in the grid
    int stride = blockDim.x * gridDim.x;

    // Each thread handles multiple elements, spaced by stride
    for (int i = tid; i < N; i += stride)
        c[i] = a[i] + b[i];
}

We launch with 128 blocks × 128 threads = 16 384 threads, but the array has 65 536 elements.
Each thread therefore handles roughly 4 elements via the loop. The result is verified
element-by-element on the CPU.

In [3]:
int a[N], b[N], c[N];
int *d_a, *d_b, *d_c;

cudaMalloc((void**)&d_a, N * sizeof(int));
cudaMalloc((void**)&d_b, N * sizeof(int));
cudaMalloc((void**)&d_c, N * sizeof(int));

for (int i = 0; i < N; i++) {
    a[i] = i;
    b[i] = i * i;
}

cudaMemcpy(d_a, a, N * sizeof(int), cudaMemcpyHostToDevice);
cudaMemcpy(d_b, b, N * sizeof(int), cudaMemcpyHostToDevice);

strided_add<<<128, 128>>>(d_a, d_b, d_c);

cudaMemcpy(c, d_c, N * sizeof(int), cudaMemcpyDeviceToHost);

bool ok = true;
for (int i = 0; i < N; i++) {
    if (c[i] != a[i] + b[i]) {
        printf("Mismatch at index %d: got %d, expected %d\n", i, c[i], a[i] + b[i]);
        ok = false;
        break;
    }
}
if (ok) printf("All %d elements verified correctly!\n", N);

cudaFree(d_a);
cudaFree(d_b);
cudaFree(d_c);

All 65536 elements verified correctly!
